# Drift Scenarios Visualization

This notebook visualizes the 4 drift scenarios for EvidentlyAI demonstrations:
1. **Data Drift**: Parameter distributions shift (μ, σ, γ change)
2. **Concept Drift**: Feature-label relationship changes (intermediate parameters)
3. **Gradual Drift**: Slow progression over 5 stages
4. **Sudden Drift**: Abrupt shift in signal characteristics

In [ ]:
# Import libraries
import sys

sys.path.append("..")

import json

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from src.signal_processing.feature_extractor import extract_features
from src.signal_processing.signal_models import SignalData

plotly_template = "plotly_dark"

## 1. Data Drift: Distribution Shift

In data drift, the signal parameter distributions shift while maintaining the same feature-label relationship.

In [ ]:
# Load or generate data drift scenarios
try:
    with open("../data/drift_scenarios/data_drift_scenario.json") as f:
        data_drift = json.load(f)
    baseline = data_drift["baseline"]
    drifted = data_drift["drifted"]
    print(f"✓ Loaded {len(baseline)} baseline, {len(drifted)} drifted signals")
except FileNotFoundError:
    print("⚠️  Drift scenario not found. Run: uv run python -m scripts.simulate_drift data-drift")
    print("   Generating synthetic drift data...")
    from src.signal_processing.signal_generator import generate_signal

    baseline = [
        generate_signal(
            "gaussian" if i % 2 == 0 else "lorentzian", drift_scenario="baseline", seed=i
        )
        for i in range(50)
    ]
    # Shift parameters for drift
    drifted = [
        generate_signal(
            "gaussian" if i % 2 == 0 else "lorentzian", drift_scenario="data_drift", seed=i + 10000
        )
        for i in range(50)
    ]
    baseline = [
        {"time": s.signal.time, "amplitude": s.signal.amplitude, "label": s.label, **s.metadata}
        for s in baseline
    ]
    drifted = [
        {"time": s.signal.time, "amplitude": s.signal.amplitude, "label": s.label, **s.metadata}
        for s in drifted
    ]


# Extract features for comparison
def get_features(signals):
    features = []
    for sig in signals:
        signal_data = SignalData(
            time=sig["time"],
            amplitude=sig["amplitude"],
            shape_type=sig.get("shape_type", "gaussian"),
        )
        feat = extract_features(signal_data)
        features.append({**feat, "label": sig["label"]})
    return pd.DataFrame(features)


baseline_feats = get_features(baseline)
drifted_feats = get_features(drifted)

# Plot signal overlays
fig = make_subplots(rows=1, cols=2, subplot_titles=("Baseline Signals", "Drifted Signals"))

for i, sig in enumerate(baseline[:10]):
    color = "green" if sig["label"] == 0 else "red"
    fig.add_trace(
        go.Scatter(
            x=sig["time"],
            y=sig["amplitude"],
            mode="lines",
            line=dict(color=color, width=1),
            opacity=0.4,
            showlegend=False,
        ),
        row=1,
        col=1,
    )

for i, sig in enumerate(drifted[:10]):
    color = "green" if sig["label"] == 0 else "red"
    fig.add_trace(
        go.Scatter(
            x=sig["time"],
            y=sig["amplitude"],
            mode="lines",
            line=dict(color=color, width=1),
            opacity=0.4,
            showlegend=False,
        ),
        row=1,
        col=2,
    )

fig.update_layout(title="Data Drift: Signal Comparison", height=400, template=plotly_template)
fig.show()

# Feature distribution comparison
fig2 = make_subplots(
    rows=1, cols=2, subplot_titles=("Peak Height Distribution", "SNR Distribution")
)

fig2.add_trace(
    go.Histogram(
        x=baseline_feats["peak_height"], name="Baseline", marker_color="blue", opacity=0.6
    ),
    row=1,
    col=1,
)
fig2.add_trace(
    go.Histogram(
        x=drifted_feats["peak_height"], name="Drifted", marker_color="orange", opacity=0.6
    ),
    row=1,
    col=1,
)

fig2.add_trace(
    go.Histogram(
        x=baseline_feats["snr"], name="Baseline", marker_color="blue", opacity=0.6, showlegend=False
    ),
    row=1,
    col=2,
)
fig2.add_trace(
    go.Histogram(
        x=drifted_feats["snr"], name="Drifted", marker_color="orange", opacity=0.6, showlegend=False
    ),
    row=1,
    col=2,
)

fig2.update_layout(
    title="Data Drift: Feature Distribution Changes",
    barmode="overlay",
    height=400,
    template=plotly_template,
)
fig2.show()

print("\nFeature Means Comparison:")
print(f"{'Feature':<15} {'Baseline':<10} {'Drifted':<10} {'% Change':<10}")
print("-" * 50)
for feat in ["peak_height", "snr", "fwhm", "noise_level"]:
    b_mean = baseline_feats[feat].mean()
    d_mean = drifted_feats[feat].mean()
    pct = ((d_mean - b_mean) / b_mean) * 100
    print(f"{feat:<15} {b_mean:<10.3f} {d_mean:<10.3f} {pct:>+9.1f}%")

## 2. Concept Drift: Label Relationship Changes

In concept drift, signals with intermediate parameters blur the healthy/unhealthy boundary.

In [ ]:
# Load concept drift scenario
try:
    with open("../data/drift_scenarios/concept_drift_scenario.json") as f:
        concept_drift = json.load(f)
    baseline_cd = concept_drift["baseline"]
    drifted_cd = concept_drift["drifted"]
    print(f"✓ Loaded concept drift: {len(baseline_cd)} baseline, {len(drifted_cd)} drifted")
except FileNotFoundError:
    print("⚠️  Run: uv run python -m scripts.simulate_drift concept-drift")
    baseline_cd = baseline[:20]  # Reuse from previous cell
    drifted_cd = drifted[:20]

# Extract features
baseline_cd_feats = get_features(baseline_cd)
drifted_cd_feats = get_features(drifted_cd)

# Feature space comparison
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=baseline_cd_feats["peak_height"],
        y=baseline_cd_feats["snr"],
        mode="markers",
        name="Baseline",
        marker=dict(
            size=10,
            color=baseline_cd_feats["label"],
            colorscale=[[0, "green"], [1, "red"]],
            showscale=False,
            opacity=0.7,
        ),
    )
)

fig.add_trace(
    go.Scatter(
        x=drifted_cd_feats["peak_height"],
        y=drifted_cd_feats["snr"],
        mode="markers",
        name="Concept Drift",
        marker=dict(
            size=10,
            symbol="x",
            color=drifted_cd_feats["label"],
            colorscale=[[0, "lightgreen"], [1, "pink"]],
            showscale=False,
            opacity=0.8,
        ),
    )
)

fig.update_layout(
    title="Concept Drift: Feature Space Boundary Blur",
    xaxis_title="Peak Height",
    yaxis_title="SNR",
    height=500,
    template=plotly_template,
)
fig.show()

print("\nLabel Distribution:")
print(
    f"Baseline - Healthy: {(baseline_cd_feats['label'] == 0).sum()}, "
    f"Unhealthy: {(baseline_cd_feats['label'] == 1).sum()}"
)
print(
    f"Drifted  - Healthy: {(drifted_cd_feats['label'] == 0).sum()}, "
    f"Unhealthy: {(drifted_cd_feats['label'] == 1).sum()}"
)

## 3. Gradual Drift: Progressive Change Over Time

Gradual drift shows slow degradation across 5 stages.

In [ ]:
# Load gradual drift stages
try:
    gradual_stages = []
    for stage in range(5):
        with open(f"../data/drift_scenarios/gradual_drift_stage_{stage}.json") as f:
            gradual_stages.append(json.load(f))
    print(f"✓ Loaded {len(gradual_stages)} gradual drift stages")
except FileNotFoundError:
    print("⚠️  Run: uv run python -m scripts.simulate_drift gradual")
    # Generate synthetic stages with increasing drift
    gradual_stages = [baseline[:10] for _ in range(5)]

# Extract features for each stage
stage_features = [get_features(stage) for stage in gradual_stages]

# Plot progression
fig = make_subplots(rows=1, cols=5, subplot_titles=[f"Stage {i}" for i in range(5)])

for col, stage_df in enumerate(stage_features, start=1):
    fig.add_trace(
        go.Scatter(
            x=stage_df["peak_height"],
            y=stage_df["snr"],
            mode="markers",
            marker=dict(
                size=8,
                color=stage_df["label"],
                colorscale=[[0, "green"], [1, "red"]],
                showscale=False,
            ),
            showlegend=False,
        ),
        row=1,
        col=col,
    )

fig.update_layout(
    title="Gradual Drift: 5-Stage Progression",
    height=400,
    template=plotly_template,
)
fig.update_xaxes(title_text="Peak Height")
fig.update_yaxes(title_text="SNR", col=1)
fig.show()

# Feature evolution over stages
fig2 = go.Figure()

for feat in ["peak_height", "snr", "fwhm", "noise_level"]:
    means = [df[feat].mean() for df in stage_features]
    fig2.add_trace(go.Scatter(x=list(range(5)), y=means, mode="lines+markers", name=feat))

fig2.update_layout(
    title="Feature Evolution Across Stages",
    xaxis_title="Stage",
    yaxis_title="Feature Value",
    height=400,
    template=plotly_template,
)
fig2.show()

## 4. Sudden Drift: Abrupt Change

Sudden drift shows an immediate shift in signal characteristics.

In [ ]:
# Load sudden drift
try:
    with open("../data/drift_scenarios/sudden_drift_scenario.json") as f:
        sudden_drift = json.load(f)
    baseline_sd = sudden_drift["baseline"]
    drifted_sd = sudden_drift["drifted"]
    print(f"✓ Loaded sudden drift: {len(baseline_sd)} baseline, {len(drifted_sd)} drifted")
except FileNotFoundError:
    print("⚠️  Run: uv run python -m scripts.simulate_drift sudden")
    baseline_sd = baseline[:20]
    drifted_sd = drifted[:20]

# Extract features
baseline_sd_feats = get_features(baseline_sd)
drifted_sd_feats = get_features(drifted_sd)

# Before/after comparison
fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=("Before: Peak Height", "After: Peak Height", "Before: SNR", "After: SNR"),
    specs=[
        [{"type": "histogram"}, {"type": "histogram"}],
        [{"type": "histogram"}, {"type": "histogram"}],
    ],
)

fig.add_trace(
    go.Histogram(x=baseline_sd_feats["peak_height"], marker_color="blue", name="Before"),
    row=1,
    col=1,
)
fig.add_trace(
    go.Histogram(x=drifted_sd_feats["peak_height"], marker_color="red", name="After"), row=1, col=2
)
fig.add_trace(
    go.Histogram(x=baseline_sd_feats["snr"], marker_color="blue", showlegend=False), row=2, col=1
)
fig.add_trace(
    go.Histogram(x=drifted_sd_feats["snr"], marker_color="red", showlegend=False), row=2, col=2
)

fig.update_layout(
    title="Sudden Drift: Before vs After",
    height=600,
    template=plotly_template,
)
fig.show()

print("\nSudden Drift Statistics:")
print(f"{'Feature':<15} {'Before':<10} {'After':<10} {'Δ':<10}")
print("-" * 50)
for feat in ["peak_height", "snr", "fwhm", "noise_level"]:
    before = baseline_sd_feats[feat].mean()
    after = drifted_sd_feats[feat].mean()
    delta = after - before
    print(f"{feat:<15} {before:<10.3f} {after:<10.3f} {delta:>+9.3f}")

## Summary: Drift Scenarios

**4 Drift Types Visualized:**
1. **Data Drift**: Parameter distributions shift → Feature means change
2. **Concept Drift**: Intermediate parameters → Decision boundary blurs
3. **Gradual Drift**: 5-stage progression → Slow degradation
4. **Sudden Drift**: Abrupt change → Immediate distribution shift

These scenarios enable EvidentlyAI drift detection demonstrations in the MLOps pipeline.